# კვირა I: მსგავსება და უმოკლესი მეზობლები (Similarity & Nearest Neighbors)
**კურსი:** ხელოვნური ინტელექტი და მანქანური სწავლების საწყისები (BTU)
**სახელმძღვანელო:** Chris Albon, *Machine Learning with Python Cookbook* (1st Edition)

---
### 🎯 ლექციის დაპირება (The Promise):
> *ამ Notebook-ის დასრულებისას თქვენ გექნებათ მომუშავე კოდი, რომელიც 64-განზომილებიან სივრცეში იპოვის ხელნაწერ ციფრებს შორის მსგავსებას, ამოიცნობს მათ და გამოავლენს, რომელ ციფრებს ურევს მოდელი ერთმანეთში და რატომ.*

### 📚 დღევანდელი სტრუქტურა:
1. **NumPy Crash Course ვექტორებისთვის:** List vs Array, Shape, Element-wise ოპერაციები და `argsort`
2. **მანძილის მეტრიკები ნულიდან:** ევკლიდური მანძილი (L2) vs კოსინუსური მსგავსება
3. **მონაცემთა გაცნობა:** `load_digits` — სურათი როგორც 64-განზომილებიანი წერტილი
4. **k-Nearest Neighbors (kNN) ნულიდან** (Pure NumPy)
5. **kNN Scikit-Learn-ით** და დაპირების შესრულება (შეცდომების ვიზუალური ანალიზი)
6. **ტექსტების მსგავსება:** სიტყვების სიხშირე და კოსინუსური კუთხე

# Windows Smart App Control fallback shim (თუ სისტემაში გააქტიურებულია Smart App Control)
import sys, types
if 'sklearn.svm._liblinear' not in sys.modules:
    try:
        import sklearn.svm._liblinear
    except Exception:
        sys.modules['sklearn.svm._liblinear'] = types.ModuleType('sklearn.svm._liblinear')

import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score

print("ყველა საჭირო ბიბლიოთეკა წარმატებით ჩაიტვირთა!")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score

print("ყველა საჭირო ბიბლიოთეკა წარმატებით ჩაიტვირთა!")

## 1. NumPy Crash Course ვექტორებისთვის (Live Demo)
სანამ მანქანურ სწავლებას დავიწყებთ, გავიცნოთ ჩვენი მთავარი იარაღი — **NumPy**.
გაუშვით და დააკვირდით თითოეულ უჯრას.

### 1.1 Python List vs NumPy Array (რატომ NumPy?)
ჩვეულებრივი პითონის სიები აკეთებენ კონკატენაციას (გადაბმას), ხოლო NumPy ახდენს **ვექტორიზაციას** (ელემენტ-ელემენტ შეკრებას).

In [ ]:
# პითონის სია:
py_list1 = [1, 2]
py_list2 = [3, 4]
print("Python List შეკრება:", py_list1 + py_list2)  # [1, 2, 3, 4] -> კონკატენაცია!

# NumPy მასივი:
np_arr1 = np.array([1, 2])
np_arr2 = np.array([3, 4])
print("NumPy Array შეკრება: ", np_arr1 + np_np_arr2 if 'np_np_arr2' in locals() else np_arr1 + np_arr2)  # [4, 6] -> ვექტორული შეკრება!

### 1.2 ვექტორის შექმნა და ზომა (`.shape`)
გაითვალისწინეთ: `(3,)` არის ერთგანზომილებიანი ვექტორი. არ აურიოთ `(1, 3)` მატრიცაში!

In [ ]:
v = np.array([10, 20, 30])
print("ვექტორი:", v)
print("განზომილება (shape):", v.shape)  # (3,) ნიშნავს 1D ვექტორს 3 ელემენტით
print("ელემენტების ტიპი:", v.dtype)

### 1.3 ელემენტ-ელემენტ ოპერაციებიდან ევკლიდურ მანძილამდე
როგორ გამოითვლება ევკლიდური მანძილი $\sqrt{\sum(a_i - b_i)^2}$ ციკლის გარეშე:

In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 6, 8])

# 1. კოორდინატების სხვაობა:
diff = a - b
print("1. a - b:", diff)

# 2. კვადრატში აყვანა:
squared = diff ** 2
print("2. diff ** 2:", squared)

# 3. ჯამი:
sum_sq = np.sum(squared)
print("3. np.sum():", sum_sq)

# 4. კვადრატული ფესვი:
dist = np.sqrt(sum_sq)
print("4. np.sqrt() [ევკლიდური მანძილი]:", dist)

### 1.4 სკალარული ნამრავლი (`np.dot`) და ინდექსებით დალაგება (`np.argsort`)
* `np.dot(a, b)`: ითვლის სკალარულ ნამრავლს $\sum a_i b_i$.
* `np.argsort(arr)`: ალაგებს ზრდადობით და აბრუნებს **ინდექსებს** (სწორედ ეს გვჭირდება უახლოესი მეზობლების საპოვნელად!).

In [ ]:
# სკალარული ნამრავლი:
dot_prod = np.dot(np.array([1, 2]), np.array([3, 4]))  # 1*3 + 2*4 = 11
print("np.dot([1, 2], [3, 4]):", dot_prod)

# მანძილების მასივი 3 მეზობლამდე:
distances = np.array([15.2, 3.1, 8.4])
sorted_indices = np.argsort(distances)
print("საწყისი მანძილები:", distances)
print("დალაგებული ინდექსები (argsort):", sorted_indices)
print(f"უახლოესი მეზობლის ინდექსია #{sorted_indices[0]}, რომლის მანძილიცაა {distances[sorted_indices[0]]}")

## 2. მანძილისა და მსგავსების მეტრიკები ნულიდან
ახლა, როცა NumPy-ის ბაზისი გვაქვს, დაწერეთ მანძილის ფორმულები.

In [ ]:
def euclidean_distance(a: np.ndarray, b: np.ndarray) -> float:
    """
    TODO 1: გამოთვალეთ ევკლიდური (L2) მანძილი ორ 1D ვექტორს შორის.
    ფორმულა: sqrt(sum((a - b) ** 2))
    """
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    
    # >>> თქვენი კოდი აქ <<<
    # distance = np.sqrt(np.sum((a - b) ** 2))
    # return float(distance)
    pass


def cosine_similarity_scratch(a: np.ndarray, b: np.ndarray) -> float:
    """
    TODO 2: გამოთვალეთ კოსინუსური მსგავსება ორ 1D ვექტორს შორის.
    ფორმულა: dot(a, b) / (norm(a) * norm(b))
    დაამატეთ შემოწმება ნულზე გაყოფის თავიდან ასაცილებლად.
    """
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    
    # >>> თქვენი კოდი აქ <<<
    # norm_prod = np.linalg.norm(a) * np.linalg.norm(b)
    # if norm_prod == 0:
    #     return 0.0
    # return float(np.dot(a, b) / norm_prod)
    pass

# შემოწმება (როცა TODO-ებს შეავსებთ, მოხსენით კომენტარები):
v1 = np.array([1, 2, 3])
v2 = np.array([2, 4, 6])  # კოლინეარული ვექტორი (2 * v1)

# print("Euclidean distance:", euclidean_distance(v1, v2))
# print("Cosine similarity (უნდა იყოს 1.0!):", cosine_similarity_scratch(v1, v2))

## 3. მონაცემთა ნაკრების გაცნობა (`load_digits`, Cookbook 2.1)
ჩავტვირთოთ 8x8 ხელით ნაწერი ციფრების ბაზა. თითოეული სურათი არის 64 პიქსელი $\to$ 64-განზომილებიანი ვექტორი!

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target

print(f"X-ის ზომა: {X.shape} (სულ {X.shape[0]} სურათი, თითოეული 64 პიქსელი)")
print(f"y-ის კლასები: {np.unique(y)}")

# დავხატოთ პირველი 4 ციფრი:
fig, axes = plt.subplots(1, 4, figsize=(8, 2))
for i, ax in enumerate(axes):
    ax.imshow(digits.images[i], cmap='gray')
    ax.set_title(f"True Label: {y[i]}")
    ax.axis('off')
plt.show()

### ℹ️ მონაცემთა დაყოფა (Train / Test Split)
> **შენიშვნა:** ქვემოთ მოცემული ფუნქცია `train_test_split` მონაცემებს ყოფს 80% სასწავლო (Train) და 20% სატესტო (Test) ნაწილებად.  
> რატომ არის ეს დაყოფა აუცილებელი, რა არის გადავარჯიშება (Overfitting) და როგორ ვაფასებთ მოდელს — **ამას დეტალურად შევისწავლით მომდევნო (II) კვირაში**.  
> ახლა ეს კოდი მიიღეთ როგორც მზა ინსტრუმენტი მოდელის ობიექტურად გამოსაცდელად.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"სასწავლო ნიმუშები (X_train): {X_train.shape[0]}")
print(f"სატესტო ნიმუშები (X_test):   {X_test.shape[0]}")

## 4. k-Nearest Neighbors (kNN) ნულიდან

ალგორითმის 4 ნაბიჯი:
1. გამოთვალეთ მანძილი $x$-დან `self.X_train`-ის ყველა წერტილამდე.
2. დაალაგეთ ინდექსები `np.argsort`-ით და აიღეთ პირველი `k` ინდექსი.
3. ამოიღეთ ამ ინდექსების შესაბამისი იარლიყები (`self.y_train`).
4. იპოვეთ ყველაზე ხშირი კლასი (`Counter(labels).most_common(1)[0][0]`).

In [ ]:
class KNNClassifierScratch:
    def __init__(self, k: int = 3):
        self.k = k
        self.X_train = None
        self.y_train = None

    def fit(self, X: np.ndarray, y: np.ndarray):
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y)
        return self

    def _predict_single(self, x: np.ndarray):
        """
        TODO 3: გამოთვალეთ მანძილები, იპოვეთ k უახლოესი ინდექსი და დააბრუნეთ უმრავლესობის კლასი.
        """
        # 1. ვითვლით მანძილებს x-სა და ყველა სასწავლო წერტილს შორის:
        # distances = np.sqrt(np.sum((self.X_train - x) ** 2, axis=1))
        
        # 2. ვირჩევთ k უახლოესი მეზობლის ინდექსს:
        # k_indices = np.argsort(distances)[:self.k]
        
        # 3. ვიღებთ მათ კლასებს:
        # k_labels = self.y_train[k_indices]
        
        # 4. უმრავლესობის კენჭისყრა:
        # return Counter(k_labels).most_common(1)[0][0]
        pass

    def predict(self, X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=float)
        return np.array([self._predict_single(x) for x in X])

## 5. მოდელის გაშვება და შედარება Scikit-Learn-თან (Cookbook 15.2)

In [ ]:
# TODO 4: გაუშვით თქვენი Scratch kNN და Scikit-Learn-ის KNeighborsClassifier(n_neighbors=3).
# შეადარეთ შედეგები (ორივე უნდა იყოს ~98.6%):

# knn_scratch = KNNClassifierScratch(k=3)
# knn_scratch.fit(X_train, y_train)
# y_pred_scratch = knn_scratch.predict(X_test)
# print("Scratch kNN Accuracy:", accuracy_score(y_test, y_pred_scratch))

# knn_sklearn = KNeighborsClassifier(n_neighbors=3)
# knn_sklearn.fit(X_train, y_train)
# y_pred_sklearn = knn_sklearn.predict(X_test)
# print("Sklearn kNN Accuracy:", accuracy_score(y_test, y_pred_sklearn))

## 6. დაპირების შესრულება: შეცდომების ანალიზი
> *„რომელ ციფრებს ურევს მოდელი ერთმანეთში და რატომ?“*

In [ ]:
# TODO 5: იპოვეთ შეცდომები:
# misclassified = np.where(y_test != y_pred_sklearn)[0]
# print(f"სულ შეცდომა: {len(misclassified)} / {len(y_test)}")

# შევხედოთ პირველ შეცდომას (მაგ. ინდექსი 51):
# err_idx = misclassified[0]
# print(f"სატესტო ინდექსი #{err_idx}: რეალურია {y_test[err_idx]}, მოდელმა თქვა {y_pred_sklearn[err_idx]}")

# დავხატოთ ეს ციფრი:
# plt.imshow(X_test[err_idx].reshape(8, 8), cmap='gray')
# plt.title(f"True: {y_test[err_idx]}, Predicted: {y_pred_sklearn[err_idx]}")
# plt.axis('off')
# plt.show()

## 7. ტექსტების მსგავსება (Cookbook 6.8)
ჯერ ვნახოთ 3-სიტყვიანი ლექსიკონი ხელით, შემდეგ კი Scikit-Learn-ის `CountVectorizer`.

In [ ]:
docs = [
    "Machine learning algorithms learn patterns from data",
    "Deep learning and machine learning use data to recognize patterns",
    "Georgian traditional cuisine features cheese and bread khachapuri"
]

cv = CountVectorizer()
X_bow = cv.fit_transform(docs).toarray()

print("ლექსიკონი (სულ", len(cv.get_feature_names_out()), "სიტყვა):")
print(cv.get_feature_names_out())

# TODO 6: გამოთვალეთ კოსინუსური მსგავსება Doc 1 vs Doc 2 და Doc 1 vs Doc 3.
# sim_1_2 = cosine_similarity_scratch(X_bow[0], X_bow[1])
# sim_1_3 = cosine_similarity_scratch(X_bow[0], X_bow[2])
# print(f"Doc 1 vs Doc 2 (ტექნიკური): {sim_1_2:.4f}")
# print(f"Doc 1 vs Doc 3 (ტექნიკური vs კულინარია): {sim_1_3:.4f}")